# BioRAG-X — Notebook 11: Final Evaluation + Ablation Framework

## Purpose

Notebook 11 is the **evaluation control center** for BioRAG-X.

It consolidates the evidence collected throughout Notebooks 01–10 into a reproducible framework for answering:

> **Which BioRAG-X configuration is actually best, for which question types, at what quality/latency/cost trade-off, and why?**

The notebook is intentionally designed as a **benchmark harness**, not as a notebook that assumes one architecture will win.

### Core evaluation layers

1. **Retrieval**
2. **BioASQ answer quality**
3. **RAG quality**
4. **Citation / grounding**
5. **Chunking**
6. **Routing**
7. **Recovery**
8. **Retrieval help / harm**
9. **ANN efficiency**
10. **Agent efficiency**
11. **Latency / token / cost**
12. **Statistical uncertainty**
13. **Failure analysis**
14. **Final system leaderboard**

### Scientific rule

No system should be declared "best" from one metric.

The leaderboard therefore reports:
- quality,
- grounding,
- retrieval,
- robustness,
- efficiency,
- and failure rates

as separate dimensions, plus an optional **research scorecard** whose weights are explicitly configurable.


## Experimental principle: one benchmark, many controlled comparisons

All system variants should be evaluated on the **same query set**, with:
- identical benchmark splits,
- identical gold evidence,
- identical answer normalization,
- fixed seeds where applicable,
- identical candidate pools when comparing downstream components,
- explicit versioning of corpus/index/chunking/model configuration.

This avoids hidden advantages caused by using different data, different retrieval depths, or different evaluator settings.


In [ ]:
from pathlib import Path
import json, math, re, time, hashlib, random, statistics
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

ROOT = Path("/mnt/data")
RUN_DIR = ROOT / "biorag_x_notebook11"
RUN_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Run directory:", RUN_DIR)
print("Seed:", RANDOM_SEED)


## 1. Metric registry

The evaluation registry separates metrics into families.

### Retrieval
- Hit@K
- Recall@K
- MRR
- nDCG@K
- MAP
- Multi-passage Recall

### BioASQ / answer
- Exact Match
- Token Precision / Recall / F1
- Lenient-style matching
- Semantic answer similarity when an embedding/evaluator is configured

### RAG
- Answer correctness
- Answer relevance
- Context precision
- Context recall
- Faithfulness / groundedness
- Unsupported claim rate

### Citation
- Citation precision
- Citation recall
- Citation support rate

### Agent / adaptive system
- Chunk Selection Accuracy
- Chunking Regret
- Retrieval Decision Accuracy
- Routing Regret
- Recovery Success Rate
- Retrieval Help Rate
- Retrieval Harm Rate
- Unnecessary Retrieval Rate
- Agent Efficiency

### ANN
- ANN Recall Loss
- ANN Speedup
- Query latency
- Memory / index size

### Production
- p50 latency
- p95 latency
- throughput
- token usage
- estimated cost / request


In [ ]:
METRIC_DIRECTIONS = {
    # higher is better
    "hit@k": "max", "recall@k": "max", "mrr": "max", "ndcg@k": "max", "map": "max",
    "exact_match": "max", "token_f1": "max", "answer_correctness": "max",
    "answer_relevance": "max", "context_precision": "max", "context_recall": "max",
    "faithfulness": "max", "citation_precision": "max", "citation_recall": "max",
    "citation_support_rate": "max", "recovery_success_rate": "max",
    "retrieval_help_rate": "max", "agent_efficiency": "max", "ann_speedup": "max",
    "chunk_selection_accuracy": "max", "retrieval_decision_accuracy": "max",
    "ann_recall_loss": "min", "chunking_regret": "min", "routing_regret": "min",
    "retrieval_harm_rate": "min", "unsupported_claim_rate": "min",
    "unnecessary_retrieval_rate": "min", "latency_p50_ms": "min",
    "latency_p95_ms": "min", "token_cost": "min", "estimated_cost": "min",
}
print("Registered metrics:", len(METRIC_DIRECTIONS))


## 2. Benchmark adapter

The notebook consumes artifacts from the previous notebooks when they exist.

Expected logical tables include:
- questions / answers / gold passage IDs
- retrieval runs
- evidence selections
- generation outputs
- citation validation
- ANN benchmark results
- agent traces
- chunking experiment results

The adapter is intentionally tolerant of column naming differences because earlier notebooks may have been run at different times.


In [ ]:
def find_artifact(patterns):
    for pattern in patterns:
        hits = sorted(ROOT.glob(pattern))
        if hits:
            return hits[0]
    return None

artifact_candidates = {
    "qa": [
        ROOT / "biorag_x_canonical" / "qa.parquet",
        ROOT / "biorag_x_canonical" / "qa_gold.parquet",
    ],
    "selection": [
        ROOT / "biorag_x_notebook10" / "selection_metrics.csv",
    ],
    "oracle": [
        ROOT / "biorag_x_notebook10" / "oracle_decomposition.csv",
    ],
    "sweep": [
        ROOT / "biorag_x_notebook10" / "context_sweep.csv",
    ],
    "failures": [
        ROOT / "biorag_x_notebook10" / "failure_attribution.csv",
    ],
    "ablation": [
        ROOT / "biorag_x_notebook10" / "ablation.csv",
    ],
    "trace": [
        ROOT / "biorag_x_notebook10" / "trace_examples.json",
    ],
}

resolved = {}
for key, paths in artifact_candidates.items():
    p = next((x for x in paths if x.exists()), None)
    resolved[key] = p
    print(f"{key:12s} -> {p}")


## 3. Canonical query table

The evaluation unit is a single benchmark question.

Each row should carry:
- question ID
- question
- reference answer(s)
- gold evidence passage IDs
- answer type
- question complexity
- optional domain metadata

Question stratification is essential because a retrieval method can look strong globally while failing badly on multi-hop or list questions.


In [ ]:
def normalize_list_value(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(v) for v in x]
    if isinstance(x, str):
        s = x.strip()
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                return [str(v) for v in obj]
        except Exception:
            pass
        return [v.strip() for v in re.split(r"[;,|]", s) if v.strip()]
    return [str(x)]

qa_path = resolved["qa"]
if qa_path is not None:
    try:
        qa_df = pd.read_parquet(qa_path)
    except Exception as exc:
        print("Could not read QA parquet:", exc)
        qa_df = pd.DataFrame()
else:
    qa_df = pd.DataFrame()

def first_present(row, names, default=None):
    for name in names:
        if name in row.index and pd.notna(row[name]):
            return row[name]
    return default

if len(qa_df):
    benchmark_df = pd.DataFrame({
        "question_id": [str(first_present(r, ["question_id","id","qid"], f"Q{i}")) for i, (_, r) in enumerate(qa_df.iterrows())],
        "question": [str(first_present(r, ["question","query","body"], "")) for _, r in qa_df.iterrows()],
        "reference_answer": [first_present(r, ["gold_answer","answer","ideal_answer","reference_answer"], "") for _, r in qa_df.iterrows()],
        "gold_passage_ids": [normalize_list_value(first_present(r, ["gold_passage_ids","gold_passages","relevant_passage_ids"], [])) for _, r in qa_df.iterrows()],
        "answer_type": [first_present(r, ["answer_type","type"], None) for _, r in qa_df.iterrows()],
    })
else:
    benchmark_df = pd.DataFrame([
        {
            "question_id": "DEMO-1",
            "question": "What is the relationship between aspirin and platelet aggregation?",
            "reference_answer": "Aspirin inhibits platelet aggregation by reducing thromboxane A2 production.",
            "gold_passage_ids": ["P-DEMO-1", "P-DEMO-2"],
            "answer_type": "summary",
        },
        {
            "question_id": "DEMO-2",
            "question": "Which genes are major hereditary breast and ovarian cancer susceptibility genes?",
            "reference_answer": "BRCA1 and BRCA2 are major susceptibility genes.",
            "gold_passage_ids": ["P-DEMO-3"],
            "answer_type": "list",
        },
    ])
    print("WARNING: using DEMO benchmark rows.")

print("Benchmark questions:", len(benchmark_df))


## 4. Shared retrieval evaluation primitives

For a ranked list and a set of gold passage IDs:

- **Hit@K** asks whether at least one gold passage appears.
- **Recall@K** asks how much of the gold evidence set was recovered.
- **MRR** rewards the rank of the first relevant passage.
- **nDCG@K** handles graded relevance if judgments are available.
- **Multi-passage recall** is critical for biomedical questions where no single passage is sufficient.


In [ ]:
def hit_at_k(ranked_ids, gold_ids, k=5):
    gold = set(map(str, gold_ids))
    return float(any(str(x) in gold for x in ranked_ids[:k])) if gold else np.nan

def recall_at_k(ranked_ids, gold_ids, k=5):
    gold = set(map(str, gold_ids))
    if not gold:
        return np.nan
    top = set(map(str, ranked_ids[:k]))
    return len(top & gold) / len(gold)

def reciprocal_rank(ranked_ids, gold_ids):
    gold = set(map(str, gold_ids))
    for i, x in enumerate(ranked_ids, 1):
        if str(x) in gold:
            return 1.0 / i
    return 0.0

def dcg(relevances):
    return sum((2**rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevances))

def ndcg_at_k(ranked_ids, gold_ids, k=10):
    gold = set(map(str, gold_ids))
    if not gold:
        return np.nan
    rel = [1 if str(x) in gold else 0 for x in ranked_ids[:k]]
    ideal = sorted(rel, reverse=True)
    denom = dcg(ideal)
    return dcg(rel) / denom if denom else 0.0

def evaluate_retrieval_run(df, id_col="question_id", ranked_col="ranked_ids", gold_col="gold_passage_ids", ks=(1,5,10,20)):
    rows = []
    for _, r in df.iterrows():
        row = {"question_id": r[id_col]}
        ranked = normalize_list_value(r[ranked_col])
        gold = normalize_list_value(r[gold_col])
        for k in ks:
            row[f"hit@{k}"] = hit_at_k(ranked, gold, k)
            row[f"recall@{k}"] = recall_at_k(ranked, gold, k)
            row[f"ndcg@{k}"] = ndcg_at_k(ranked, gold, k)
        row["mrr"] = reciprocal_rank(ranked, gold)
        rows.append(row)
    return pd.DataFrame(rows)


## 5. BioASQ-style answer evaluation

Biomedical QA requires more than exact string match.

This baseline provides:
- Exact Match after normalization
- token precision / recall / F1
- synonym-aware matching hook
- semantic similarity hook

For actual BioASQ reporting, use the benchmark's official answer evaluation where the precise task/type requirements apply. This notebook keeps the internal evaluation interface modular so an official scorer can replace the baseline.


In [ ]:
import unicodedata

def normalize_answer(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def token_set(text):
    return set(normalize_answer(text).split())

def token_prf(pred, ref):
    p = normalize_answer(pred).split()
    r = normalize_answer(ref).split()
    if not p or not r:
        return {"precision": float(p == r), "recall": float(p == r), "f1": float(p == r)}
    common = len(set(p) & set(r))
    precision = common / len(set(p))
    recall = common / len(set(r))
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

def exact_match(pred, ref):
    return float(normalize_answer(pred) == normalize_answer(ref))

def evaluate_answers(pred_df, answer_col="prediction", ref_col="reference_answer", id_col="question_id"):
    rows = []
    for _, r in pred_df.iterrows():
        m = token_prf(r[answer_col], r[ref_col])
        rows.append({
            "question_id": r[id_col],
            "exact_match": exact_match(r[answer_col], r[ref_col]),
            "token_precision": m["precision"],
            "token_recall": m["recall"],
            "token_f1": m["f1"],
        })
    return pd.DataFrame(rows)


## 6. RAG and citation metrics

Generation evaluation should be joined with evidence evaluation.

The important distinction is:

> **A correct answer is not necessarily a grounded answer.**

Track answer correctness separately from:
- context precision
- context recall
- faithfulness
- citation precision
- citation recall
- citation support rate
- unsupported claim rate

For model-based judges, store:
- evaluator model/version
- prompt/version
- raw judgment
- score
- uncertainty


In [ ]:
def safe_mean(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return float(s.mean()) if len(s) else np.nan

def summarize_numeric(df, columns):
    return {c: safe_mean(df[c]) for c in columns if c in df.columns}


## 7. Oracle decomposition and retrieval HELPED / HURT

Notebook 10 established:

- **No Retrieval**
- **Oracle Retrieval**
- **Actual Retrieval**

Notebook 11 aggregates these into system-level diagnostics.

Definitions:

**Retrieval Gap = Oracle − Actual**

**Generation Gain = Actual − No Retrieval**

**Retrieval Help Rate** = fraction of questions where Actual > No Retrieval.

**Retrieval Harm Rate** = fraction where Actual < No Retrieval.

These metrics diagnose whether the retrieval stack is genuinely useful.


In [ ]:
oracle_path = resolved["oracle"]
if oracle_path is not None and oracle_path.exists():
    oracle_df = pd.read_csv(oracle_path)
else:
    oracle_df = pd.DataFrame(columns=[
        "question_id", "oracle_vs_actual_support_gap", "actual_vs_no_retrieval_gain", "retrieval_effect"
    ])

def aggregate_oracle(df):
    if len(df) == 0:
        return {
            "retrieval_gap_mean": np.nan,
            "generation_gain_mean": np.nan,
            "retrieval_help_rate": np.nan,
            "retrieval_harm_rate": np.nan,
        }
    gap = pd.to_numeric(df.get("oracle_vs_actual_support_gap"), errors="coerce")
    gain = pd.to_numeric(df.get("actual_vs_no_retrieval_gain"), errors="coerce")
    effect = df.get("retrieval_effect", pd.Series(dtype=str))
    return {
        "retrieval_gap_mean": float(gap.mean()) if len(gap.dropna()) else np.nan,
        "generation_gain_mean": float(gain.mean()) if len(gain.dropna()) else np.nan,
        "retrieval_help_rate": float((effect == "HELPED").mean()) if len(effect) else np.nan,
        "retrieval_harm_rate": float((effect == "HURT").mean()) if len(effect) else np.nan,
    }

oracle_summary = aggregate_oracle(oracle_df)
oracle_summary


## 8. Chunking evaluation and Chunking Regret

A chunking method should be judged on downstream retrieval/answer utility, not only chunk statistics.

Define an **oracle chunking choice** per question as the best-performing candidate method under a fixed evaluation budget.

Then:

**Chunking Regret(q) = Utility(best chunker) − Utility(selected chunker)**

Aggregate:
- mean regret
- median regret
- regret by question type
- selection accuracy

This directly evaluates the adaptive chunking router from Notebooks 05 and 09.


In [ ]:
def compute_regret_table(df, group_col="question_id", choice_col="chunking_method", utility_col="utility"):
    required = {group_col, choice_col, utility_col}
    if not required.issubset(df.columns):
        return pd.DataFrame()
    oracle = df.groupby(group_col)[utility_col].max().rename("oracle_utility")
    chosen = df.set_index(group_col).groupby(level=0)[utility_col].first().rename("chosen_utility")
    out = pd.concat([oracle, chosen], axis=1)
    out["chunking_regret"] = out["oracle_utility"] - out["chosen_utility"]
    return out.reset_index()

# Optional adapter: users can point this to the experiment table emitted in Notebook 03–05.
chunking_path = find_artifact([
    "**/*chunking*selection*.csv",
    "**/*chunking*benchmark*.csv",
    "**/*chunking*.parquet",
])
if chunking_path:
    print("Chunking artifact candidate:", chunking_path)
else:
    print("No chunking benchmark artifact discovered yet.")


## 9. Routing Regret

For each query, compare the agent's selected retrieval path against the best available retrieval strategy **under the same benchmark conditions**.

Example:

```text
Question q
  BM25 utility = 0.61
  Dense utility = 0.70
  Hybrid utility = 0.75  ← oracle best
  Graph utility = 0.73
  Agent chose Graph = 0.73

Routing Regret = 0.75 − 0.73 = 0.02
```

Do not compare systems with different retrieval budgets unless cost is explicitly included in the utility definition.


In [ ]:
routing_path = find_artifact([
    "**/*routing*benchmark*.csv",
    "**/*retrieval*decision*.csv",
    "**/*agent*benchmark*.csv",
])

if routing_path:
    routing_df = pd.read_csv(routing_path) if routing_path.suffix.lower() == ".csv" else pd.read_parquet(routing_path)
    print("Routing columns:", routing_df.columns.tolist())
else:
    routing_df = pd.DataFrame()
    print("No routing benchmark artifact discovered yet.")

def routing_regret(df, query_col="question_id", method_col="method", utility_col="utility", chosen_col="chosen"):
    if not {query_col, method_col, utility_col}.issubset(df.columns):
        return pd.DataFrame()
    oracle = df.groupby(query_col)[utility_col].max().rename("oracle_utility")
    chosen = df[df.get(chosen_col, False) == True].groupby(query_col)[utility_col].first().rename("chosen_utility")
    out = pd.concat([oracle, chosen], axis=1)
    out["routing_regret"] = out["oracle_utility"] - out["chosen_utility"]
    return out.reset_index()

routing_regret_df = routing_regret(routing_df)


## 10. Recovery success

Agentic retrieval should not get credit merely for making another tool call.

A recovery is successful only when the extra retrieval:
1. moves the evidence state from insufficient → sufficient, or
2. materially improves a validated answer/evidence metric,
3. without violating a predefined cost/latency budget.

This avoids rewarding unnecessary agent loops.


In [ ]:
def recovery_success_rate(trace_df, recovered_col="recovery_attempted",
                           before_col="before_score", after_col="after_score",
                           min_gain=0.02):
    if not {recovered_col, before_col, after_col}.issubset(trace_df.columns):
        return np.nan
    attempted = trace_df[trace_df[recovered_col] == True]
    if len(attempted) == 0:
        return np.nan
    gain = pd.to_numeric(attempted[after_col], errors="coerce") - pd.to_numeric(attempted[before_col], errors="coerce")
    return float((gain >= min_gain).mean())

print("Recovery metric function ready.")


## 11. ANN evaluation

Approximate nearest-neighbor search introduces a deliberate quality/latency trade-off.

Measure:

**ANN Recall Loss = Exact Recall@K − ANN Recall@K**

**ANN Speedup = Exact latency / ANN latency**

A valid ANN configuration should be compared against an **exact-search reference** using the same embeddings, query set, and candidate definition.

Do not compare ANN recall from one embedding model against exact recall from a different embedding model.


In [ ]:
def ann_metrics(exact_recall, ann_recall, exact_latency_ms, ann_latency_ms):
    recall_loss = exact_recall - ann_recall
    speedup = exact_latency_ms / ann_latency_ms if ann_latency_ms and ann_latency_ms > 0 else np.nan
    return {
        "ann_recall_loss": recall_loss,
        "ann_speedup": speedup,
    }

def aggregate_ann_rows(rows):
    if not rows:
        return {}
    df = pd.DataFrame(rows)
    return {
        "ann_recall_loss_mean": safe_mean(df["ann_recall_loss"]),
        "ann_speedup_mean": safe_mean(df["ann_speedup"]),
    }

print("ANN evaluation functions ready.")


## 12. Agent efficiency

Agentic retrieval has two dimensions:

**Quality**
- answer quality
- retrieval recall
- grounding

**Efficiency**
- number of retrieval calls
- recovery rounds
- tool diversity
- latency
- token usage
- estimated cost

A useful normalized metric is:

**Agent Efficiency = validated utility / total compute cost**

The exact cost model is configurable:
- latency only,
- token-weighted,
- cloud/API dollars,
- or a weighted operational cost index.


In [ ]:
def agent_efficiency(utility, latency_ms=None, tokens=None, estimated_cost=None,
                     latency_weight=1.0, token_weight=0.001, cost_weight=1.0):
    denom = 0.0
    if latency_ms is not None:
        denom += latency_weight * max(float(latency_ms), 0.0)
    if tokens is not None:
        denom += token_weight * max(float(tokens), 0.0)
    if estimated_cost is not None:
        denom += cost_weight * max(float(estimated_cost), 0.0)
    return float(utility) / denom if denom > 0 else np.nan

def p50(values):
    return float(np.percentile(values, 50)) if len(values) else np.nan

def p95(values):
    return float(np.percentile(values, 95)) if len(values) else np.nan


## 13. Latency and cost accounting

Break total latency into stages:

```text
ingestion / indexing  [offline]
query analysis        [online]
BM25                  [online]
dense ANN             [online]
fusion                [online]
reranker              [online]
graph / PageIndex     [online]
evidence selection    [online]
generation            [online]
citation validation   [online]
repair / regeneration [conditional]
```

For production analysis, report both:
- **p50**
- **p95**

and separate:
- first-pass requests,
- recovered requests,
- abstained requests.


In [ ]:
def latency_summary(df, latency_col="latency_ms", group_col="system"):
    if not {latency_col, group_col}.issubset(df.columns):
        return pd.DataFrame()
    rows = []
    for name, g in df.groupby(group_col):
        vals = pd.to_numeric(g[latency_col], errors="coerce").dropna().values
        rows.append({
            "system": name,
            "n": len(vals),
            "latency_p50_ms": p50(vals),
            "latency_p95_ms": p95(vals),
            "latency_mean_ms": float(np.mean(vals)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)


## 14. Bootstrap confidence intervals

Point estimates are insufficient when comparing retrieval systems.

This notebook provides non-parametric bootstrap confidence intervals over **questions**, preserving each question as the independent unit.

Default:
- 2,000 bootstrap samples
- percentile 95% CI

For paired comparisons, bootstrap the **per-question metric difference**, which is usually more informative than bootstrapping each system independently.


In [ ]:
def bootstrap_ci(values, statistic=np.mean, n_boot=2000, seed=42, alpha=0.05):
    vals = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(vals) == 0:
        return {"estimate": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": 0}
    rng = np.random.default_rng(seed)
    estimate = float(statistic(vals))
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boot[i] = statistic(sample)
    return {
        "estimate": estimate,
        "ci_low": float(np.quantile(boot, alpha/2)),
        "ci_high": float(np.quantile(boot, 1-alpha/2)),
        "n": int(len(vals)),
    }

def paired_bootstrap_ci(a, b, n_boot=2000, seed=42):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    diff = a[mask] - b[mask]
    return bootstrap_ci(diff, n_boot=n_boot, seed=seed)

print(bootstrap_ci([0.5,0.6,0.7,0.8,0.9]))


## 15. Statistical comparison

For two systems on the same benchmark questions, prefer **paired** analysis.

Recommended reporting:
- mean metric difference
- 95% bootstrap CI
- effect direction
- sample size
- optionally Wilcoxon signed-rank / permutation test

Avoid declaring victory because a score differs by a tiny amount without uncertainty analysis.


In [ ]:
def paired_comparison(df, system_a, system_b, metric, system_col="system", qid_col="question_id"):
    pivot = df.pivot_table(index=qid_col, columns=system_col, values=metric, aggfunc="mean")
    if system_a not in pivot.columns or system_b not in pivot.columns:
        return {}
    out = paired_bootstrap_ci(pivot[system_a].values, pivot[system_b].values)
    out.update({
        "system_a": system_a,
        "system_b": system_b,
        "metric": metric,
    })
    return out


## 16. Failure taxonomy aggregation

Final evaluation should report not only "how many failed" but **where failures occurred**.

BioRAG-X taxonomy:

| ID | Layer |
|---|---|
| F01 | ingestion |
| F02 | chunking |
| F03 | entity extraction |
| F04 | lexical miss |
| F05 | dense miss |
| F06 | ANN recall loss |
| F07 | fusion |
| F08 | reranker |
| F09 | graph linking |
| F10 | graph traversal |
| F11 | PageIndex |
| F12 | query expansion drift |
| F13 | evidence incompleteness |
| F14 | evidence conflict |
| F15 | generation error |
| F16 | hallucination |
| F17 | citation failure |
| F18 | should-have-abstained |
| F19 | unnecessary retrieval |
| F20 | latency/cost |


In [ ]:
FAILURE_LABELS = {
    "F01":"ingestion","F02":"chunking","F03":"entity extraction","F04":"lexical miss",
    "F05":"dense miss","F06":"ANN recall loss","F07":"fusion","F08":"reranker",
    "F09":"graph linking","F10":"graph traversal","F11":"PageIndex","F12":"query expansion drift",
    "F13":"evidence incompleteness","F14":"evidence conflict","F15":"generation error",
    "F16":"hallucination","F17":"citation failure","F18":"should-have-abstained",
    "F19":"unnecessary retrieval","F20":"latency/cost",
}

failure_path = resolved["failures"]
if failure_path is not None and failure_path.exists():
    failure_df = pd.read_csv(failure_path)
else:
    failure_df = pd.DataFrame()

def explode_failures(df, col="failure_tags"):
    if len(df) == 0 or col not in df.columns:
        return pd.DataFrame(columns=["failure_code","failure_name","count"])
    rows = []
    for _, r in df.iterrows():
        tags = normalize_list_value(r[col])
        for t in tags:
            rows.append({"failure_code": t, "failure_name": FAILURE_LABELS.get(t, t)})
    out = pd.DataFrame(rows)
    return out.groupby(["failure_code","failure_name"]).size().reset_index(name="count").sort_values("count", ascending=False)

failure_summary_df = explode_failures(failure_df)
failure_summary_df


## 17. System registry

Every configuration should have a stable, human-readable system ID.

Example:

```text
S01  BM25
S02  Dense-MedCPT
S03  Hybrid-RRF
S04  Hybrid-RRF + Reranker
S05  GraphRAG
S06  PageIndex
S07  Static Multi-Tool
S08  Adaptive Agentic
S09  Adaptive Agentic + Citation Guard
```

The registry prevents accidental comparison of a partially configured system against a fully configured one.


In [ ]:
SYSTEM_REGISTRY = pd.DataFrame([
    ["S01", "BM25", "lexical"],
    ["S02", "Dense-MedCPT", "dense"],
    ["S03", "Hybrid-RRF", "hybrid"],
    ["S04", "Hybrid-RRF+Reranker", "hybrid_reranked"],
    ["S05", "GraphRAG", "graph"],
    ["S06", "PageIndex", "pageindex"],
    ["S07", "Static Multi-Tool", "static_multitool"],
    ["S08", "Adaptive Agentic", "agentic"],
    ["S09", "Adaptive Agentic+CiteGuard", "agentic_grounded"],
], columns=["system_id","system_name","family"])

SYSTEM_REGISTRY


## 18. Metric normalization for the leaderboard

Metrics have different directions and scales.

Before combining metrics:
- clip bounded metrics to [0, 1]
- normalize efficiency/cost metrics explicitly
- document every weight
- never hide the raw metrics

A composite score is therefore a **decision aid**, not a replacement for the full leaderboard.


In [ ]:
def minmax(series):
    s = pd.to_numeric(series, errors="coerce")
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.ones(len(s)), index=series.index)
    return (s - lo) / (hi - lo)

def normalize_metric(series, direction="max"):
    if direction == "max":
        return minmax(series)
    if direction == "min":
        return 1.0 - minmax(series)
    raise ValueError(direction)


## 19. Final leaderboard builder

The leaderboard expects one row per system with precomputed evaluation metrics.

Default quality-first score:

- 30% answer quality
- 20% retrieval quality
- 20% grounding/citation
- 10% robustness / adaptive behavior
- 10% efficiency
- 10% operational performance

These are **default research weights only**. They should be changed for the deployment objective, and the raw metrics must remain visible.


In [ ]:
DEFAULT_WEIGHTS = {
    "answer_quality": 0.30,
    "retrieval_quality": 0.20,
    "grounding": 0.20,
    "adaptive_robustness": 0.10,
    "efficiency": 0.10,
    "operations": 0.10,
}

def build_leaderboard(metrics_df, weights=DEFAULT_WEIGHTS):
    df = metrics_df.copy()

    groups = {
        "answer_quality": ["token_f1", "answer_correctness"],
        "retrieval_quality": ["recall@10", "mrr"],
        "grounding": ["faithfulness", "citation_support_rate"],
        "adaptive_robustness": ["recovery_success_rate", "retrieval_help_rate"],
        "efficiency": ["agent_efficiency", "ann_speedup"],
        "operations": ["latency_p95_ms"],
    }

    component_scores = {}
    for group, cols in groups.items():
        available = [c for c in cols if c in df.columns]
        if not available:
            component_scores[group] = pd.Series(np.nan, index=df.index)
            continue

        normalized = []
        for c in available:
            direction = METRIC_DIRECTIONS.get(c, "max")
            normalized.append(normalize_metric(df[c], direction))
        component_scores[group] = pd.concat(normalized, axis=1).mean(axis=1)

    for group, values in component_scores.items():
        df[group] = values

    df["composite_score"] = 0.0
    total_weight = 0.0
    for group, weight in weights.items():
        if group in df.columns:
            df["composite_score"] += weight * df[group].fillna(0.0)
            total_weight += weight

    if total_weight:
        df["composite_score"] /= total_weight

    return df.sort_values("composite_score", ascending=False)

print("Leaderboard builder ready.")


## 20. Build an executable demo leaderboard

This section creates a **control-flow-only demonstration** when real multi-system benchmark results are not yet present.

It is intentionally flagged as DEMO.

Replace this table with actual outputs from the retrieval, agent, generation, and evaluation notebooks before reporting results.


In [ ]:
demo_metrics = pd.DataFrame([
    {
        "system_id":"S01","system_name":"BM25",
        "token_f1":0.55,"answer_correctness":0.58,"recall@10":0.61,"mrr":0.57,
        "faithfulness":0.70,"citation_support_rate":0.45,
        "recovery_success_rate":0.00,"retrieval_help_rate":0.52,
        "agent_efficiency":0.95,"ann_speedup":1.00,"latency_p95_ms":35,
    },
    {
        "system_id":"S03","system_name":"Hybrid-RRF",
        "token_f1":0.64,"answer_correctness":0.67,"recall@10":0.76,"mrr":0.71,
        "faithfulness":0.76,"citation_support_rate":0.61,
        "recovery_success_rate":0.00,"retrieval_help_rate":0.66,
        "agent_efficiency":0.82,"ann_speedup":4.20,"latency_p95_ms":62,
    },
    {
        "system_id":"S08","system_name":"Adaptive Agentic",
        "token_f1":0.69,"answer_correctness":0.72,"recall@10":0.81,"mrr":0.77,
        "faithfulness":0.78,"citation_support_rate":0.67,
        "recovery_success_rate":0.62,"retrieval_help_rate":0.72,
        "agent_efficiency":0.71,"ann_speedup":4.20,"latency_p95_ms":111,
    },
    {
        "system_id":"S09","system_name":"Adaptive Agentic+CiteGuard",
        "token_f1":0.68,"answer_correctness":0.72,"recall@10":0.81,"mrr":0.77,
        "faithfulness":0.90,"citation_support_rate":0.88,
        "recovery_success_rate":0.62,"retrieval_help_rate":0.72,
        "agent_efficiency":0.66,"ann_speedup":4.20,"latency_p95_ms":137,
    },
])
demo_leaderboard = build_leaderboard(demo_metrics)
demo_leaderboard[[c for c in ["system_id","system_name","answer_quality","retrieval_quality","grounding","adaptive_robustness","efficiency","operations","composite_score"] if c in demo_leaderboard.columns]]


## 21. Pareto frontier

A single composite score can hide important trade-offs.

For production decisions, inspect the Pareto frontier over:
- answer quality,
- grounding,
- p95 latency,
- cost.

A system is dominated if another system is at least as good on every chosen axis and strictly better on one.


In [ ]:
def pareto_frontier(df, maximize=("answer_quality","grounding"), minimize=("latency_p95_ms",)):
    required = list(maximize) + list(minimize)
    d = df.dropna(subset=[c for c in required if c in df.columns]).copy()
    if len(d) == 0:
        return d

    keep = []
    for i, a in d.iterrows():
        dominated = False
        for j, b in d.iterrows():
            if i == j:
                continue
            better_or_equal_max = all(b[x] >= a[x] for x in maximize if x in d.columns)
            better_or_equal_min = all(b[x] <= a[x] for x in minimize if x in d.columns)
            strictly_better = (
                any(b[x] > a[x] for x in maximize if x in d.columns) or
                any(b[x] < a[x] for x in minimize if x in d.columns)
            )
            if better_or_equal_max and better_or_equal_min and strictly_better:
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return d.loc[keep]

pareto_demo = pareto_frontier(
    demo_leaderboard,
    maximize=("answer_quality","grounding"),
    minimize=("operations",)  # normalized operational component; lower raw latency remains visible separately
)
pareto_demo[["system_id","system_name","composite_score"]]


## 22. Question-type / complexity stratification

Aggregate metrics should be broken down by:

- question type
- single-hop vs multi-hop
- number of gold passages
- evidence concentration
- lexical overlap
- graph-link availability
- difficulty bucket

This is where the adaptive system's value should become visible.

For example, a system may lose on simple factual questions but win substantially on:
- multi-passage synthesis,
- relation-heavy questions,
- weak lexical overlap,
- multi-hop biomedical questions.


In [ ]:
def add_complexity_features(df):
    out = df.copy()
    out["gold_passage_count"] = out["gold_passage_ids"].apply(lambda x: len(normalize_list_value(x)))
    out["complexity_bucket"] = pd.cut(
        out["gold_passage_count"].clip(lower=0),
        bins=[-1,1,2,999],
        labels=["single-passage","two-passage","multi-passage"]
    )
    out["question_length"] = out["question"].astype(str).str.split().str.len()
    return out

benchmark_features = add_complexity_features(benchmark_df)
benchmark_features.head()


## 23. Experimental matrix

A full BioRAG-X evaluation should compare at minimum:

### Retrieval
- BM25
- Dense / MedCPT
- Hybrid + RRF
- Hybrid + reranker
- GraphRAG
- PageIndex
- static multi-tool

### Adaptive
- routed lexical/dense
- routed graph/PageIndex
- agentic retrieval
- agentic retrieval + recovery

### End-to-end
- no retrieval
- oracle retrieval
- actual retrieval
- actual + citation guard
- actual + citation guard + abstention

Every row should record the exact configuration:
- embedding model
- chunker
- candidate K
- fusion method
- reranker
- ANN index type
- graph version
- PageIndex version/config
- generator
- evaluator
- seed


## 24. Reproducibility manifest

The final experiment manifest should be versioned with:
- corpus fingerprint
- dataset revision
- chunking version
- embedding model/version
- index parameters
- graph build version
- retrieval configuration
- agent policy
- generation model/version
- prompt version
- evaluator version
- random seed
- benchmark split
- timestamp

This prevents "same system" from silently changing between experiments.


In [ ]:
manifest = {
    "project": "BioRAG-X",
    "notebook": "11_evaluation_and_ablation",
    "random_seed": RANDOM_SEED,
    "benchmark_rows": int(len(benchmark_df)),
    "metric_registry_size": len(METRIC_DIRECTIONS),
    "system_registry_rows": len(SYSTEM_REGISTRY),
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "demo_results_must_not_be_reported": True,
}
manifest_path = RUN_DIR / "evaluation_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest_path)


## 25. Final reporting template

The final paper / project report should present results in this order:

### Table 1 — Retrieval
Hit@5 / 10 / 20, Recall@K, MRR, nDCG, multi-passage recall.

### Table 2 — End-to-end biomedical QA
BioASQ metrics, answer correctness, relevance.

### Table 3 — Grounding and citation
Faithfulness, citation precision/recall/support, unsupported claim rate.

### Table 4 — Adaptive intelligence
Chunk Selection Accuracy, Chunking Regret, Retrieval Decision Accuracy, Routing Regret, Recovery Success Rate, Help/Harm.

### Table 5 — Efficiency
p50/p95 latency, tokens, cost, ANN recall loss, ANN speedup, agent efficiency.

### Table 6 — Failure analysis
F01–F20 counts and representative failure cases.

### Figure — Quality / cost Pareto frontier

### Figure — Retrieval effect
HELPED / NEUTRAL / HURT.

### Figure — Error funnel
retrieval miss → evidence insufficiency → generation error → citation failure → abstention.


# Final interpretation rules

The project should make conclusions based on **evidence chains**, not one leaderboard number.

### Case 1 — Oracle ≫ Actual
The generation model can use the evidence, but retrieval/evidence selection is the bottleneck.

### Case 2 — Actual ≫ No Retrieval
Retrieval is adding real value.

### Case 3 — Actual ≈ No Retrieval
The question set may not need retrieval, or the retriever may not be finding useful evidence.

### Case 4 — Actual < No Retrieval
Retrieval is actively hurting. Inspect:
- noisy evidence,
- contradiction,
- context overload,
- wrong routing,
- bad chunking,
- query expansion drift.

### Case 5 — Good answer, poor citation support
The model may know the answer parametrically but the system is not demonstrating evidence grounding.

### Case 6 — Good retrieval, poor answer
The bottleneck has moved to generation, evidence selection, context reconstruction, or instruction following.

### Case 7 — High quality, unacceptable p95/cost
The configuration may be research-interesting but production-infeasible.

The strongest BioRAG-X result is therefore not:

> **"Our agentic RAG got X%."**

It is:

> **"We can quantify when adaptive retrieval helps, why it helps, what it costs, when it fails, and whether the resulting biomedical claims are actually supported by traceable evidence."**


# Handoff to Notebook 12 — Production Simulation

Notebook 12 should now take the validated evaluation outputs and simulate a production environment:

- request orchestration
- concurrent queries
- caching
- model/index warm-up
- latency budgets
- cost budgets
- observability
- trace storage
- failures/timeouts/retries
- drift monitoring
- versioned indexes
- A/B testing
- SLOs
- governance
- final production architecture

Only after Notebook 12 should the **BioRAG-X production UI** become the presentation layer over the experimentally validated system.
